# Il pattern di cio che manca riconosce il paziente

**CAF GIAS, Universita degli Studi Link, modulo 2.4** | Michele Ferramola

Dimostrazione da 5 minuti per il blocco *Gestione dei missing data*.

---

### La domanda

Un dataset clinico ha migliaia di colonne. Alcune sono piene, altre hanno dei buchi.
Se **cancello tutti i valori** e tengo solo l'informazione *c'e il dato oppure no*,
quanto riesco ancora a sapere del paziente?

### Il dato

NHANES 2013-2014, indagine pubblica statunitense su salute e nutrizione.
6.113 adulti, 1.810 variabili.
Il file che carichiamo **non contiene nessun valore clinico**: solo `True` e `False`,
piu l'etichetta del sesso che useremo come bersaglio.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

URL = "https://raw.githubusercontent.com/MicheleFerramola00/lezioni/caf-unilink-m2.4/dati/mancanza_nhanes_adulti.parquet"

M = pd.read_parquet(URL)
y = M.pop("sesso_F").values          # 1 = donna, 0 = uomo

print(f"pazienti: {len(M)}   colonne: {M.shape[1]}   donne: {y.mean():.1%}")
print(f"tipi di dato presenti nel file: {set(map(str, M.dtypes))}")
M.iloc[:6, :6]


Sopra non si legge nessun esito clinico. Solo `True` dove il dato manca e `False` dove c'e.

Teniamo le colonne che hanno un minimo di variabilita, cioe che non sono ne sempre piene ne sempre vuote.


In [ ]:
M = M.loc[:, (M.mean() > 0.01) & (M.mean() < 0.99)]
print(f"colonne con variabilita: {M.shape[1]}")

tassoF, tassoM = M[y == 1].mean(), M[y == 0].mean()
divario = (tassoF - tassoM)

top = pd.concat([divario.nsmallest(10), divario.nlargest(10)]).sort_values()
ax = top.plot.barh(figsize=(9, 6), color=["#155999" if v < 0 else "#D4A13A" for v in top])
ax.set_xlabel("quanto piu spesso il dato manca   (blu = manca agli uomini, oro = manca alle donne)")
ax.set_title("Le venti colonne in cui il buco distingue di piu i due sessi")
ax.axvline(0, color="black", lw=0.8); plt.tight_layout(); plt.show()


Nessuna sorpresa: salute riproduttiva, HPV, comportamento sessuale.
Domande che a un sesso non vengono proprio rivolte.

### Adesso la prova

Alleno una regressione logistica **sulla sola matrice dei buchi** e le chiedo di indovinare il sesso.


In [ ]:
def prova(X, etichetta):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    mod = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
    auroc = roc_auc_score(yte, mod.predict_proba(Xte)[:, 1])
    print(f"{etichetta:<46} AUROC = {auroc:.3f}   ({X.shape[1]} colonne)")
    return auroc

prova(M, "tutte le colonne di mancanza")


### L'obiezione, che e legittima

*"Certo: gravidanza, prostata, HPV. Sono domande che si fanno a un sesso solo.*
*Togli quelle e il problema sparisce."*

Proviamo. Rimuovo **ogni colonna appartenente a un dominio sesso-specifico**:
salute riproduttiva, HPV, comportamento sessuale, prostata, gravidanza, ciclo mestruale.


In [ ]:
prefissi = ("rh_", "vag_", "sx_", "imm_hpv")
parole   = ("prostat", "pregn", "menstr", "ovar", "hysterect")
sesso_specifiche = [c for c in M.columns
                    if c.lower().startswith(prefissi) or any(p in c.lower() for p in parole)]
print(f"colonne rimosse: {len(sesso_specifiche)}")
print()

prova(M, "tutte le colonne")
prova(M.drop(columns=sesso_specifiche), "senza i domini sesso-specifici")


Il modello continua a riconoscere il sesso quasi perfettamente.

### Fino a che punto regge

Alzo il livello: elimino **ogni colonna** il cui tasso di mancanza differisce fra i due sessi
piu di una certa soglia. E un filtro che nella pratica nessuno applicherebbe mai.


In [ ]:
scala = []
for soglia in (1.00, 0.30, 0.10, 0.05, 0.02):
    tenute = M.loc[:, divario.abs() <= soglia]
    scala.append((f"divario <= {soglia:.0%}", prova(tenute, f"divario di mancanza <= {soglia:.0%}")))

print()
lab = [s for s, _ in scala]; val = [v for _, v in scala]
ax = plt.figure(figsize=(8, 4)).gca()
ax.bar(lab, val, color="#103F6D"); ax.axhline(0.5, color="#D4A13A", ls="--", label="caso")
ax.set_ylim(0.4, 1.02); ax.set_ylabel("AUROC nel riconoscere il sesso")
ax.set_title("Quanto devo mutilare il dato prima che il segnale sparisca")
ax.legend(); plt.tight_layout(); plt.show()


### Dove vive davvero il segnale

Ultima verifica: e colpa delle macchine di laboratorio o delle domande del questionario?


In [ ]:
lab_fis = [c for c in M.columns if c.split("_")[0] in
           ("lab", "bp", "bm", "bmx", "chol", "gluc", "cbc", "hdl", "ldl",
            "tri", "ins", "alb", "cre", "urx", "phy", "grip", "dxa", "ohx")]
prova(M[lab_fis], "solo laboratorio ed esame fisico")


---

## Cosa abbiamo visto

1. **Il buco non e assenza di informazione.** La sola mappa di cio che manca identifica il sesso.
2. **Togliere le variabili ovvie non basta.** Il segnale e distribuito su centinaia di colonne,
   ciascuna delle quali da sola sembra innocua.
3. **Il segnale vive nel processo, non nello strumento.** Le colonne di laboratorio quasi non lo portano:
   lo porta cio che e stato chiesto, a chi, e quando.

### Perche riguarda l'imaging

In imaging il buco non e una cella vuota: e **un paziente intero che non compare nel dataset**,
perche nessuno gli ha prescritto l'esame. La prescrizione e una decisione umana, con determinanti noti:
accesso, lingua, eta, presunzione diagnostica.
Lo stesso meccanismo, con una traccia ancora piu difficile da vedere.

### La domanda da portare in gara

> **Chi manca dal vostro dataset, e per quale meccanismo di selezione?**

Ancoraggio: AI Act art. 10 (governance dei dati), STANDING Together (documentazione della diversita dei dataset).
